# Manual Multiclass Logistic Regression Classification

This notebook follows the logistic-regression implementation in `Lab3.ipynb` and the assignment document. It uses the Word2Vec document vectors and the `Category` column from `preprocessed_research_papers_with_embeddings.csv`.

Because the dataset has six categories, this notebook uses one multiclass logistic-regression model with a **softmax** output. Softmax produces one probability distribution whose values always add up to `1.0`. No built-in classifier is used.

The categories are imbalanced, so the manual cross-entropy gradient gives each category balanced influence during training.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_PATH = PROJECT_ROOT / "data" / "processed_data" / "preprocessed_research_papers_with_embeddings.csv"
VECTOR_PATH = PROJECT_ROOT / "data" / "processed_data" / "document_vectors.npy"

papers = pd.read_csv(DATASET_PATH, usecols=["Category"])
y = papers["Category"].fillna("").astype(str).str.strip().to_numpy()
X = np.asarray(np.load(VECTOR_PATH, mmap_mode="r"), dtype=np.float64)

if X.shape[0] != len(y):
    raise ValueError("Vector and label row counts do not match")
if not np.isfinite(X).all():
    raise ValueError("Vectors contain NaN or infinite values")

# Keep every category in both sets with a manual stratified split.
random_generator = np.random.default_rng(42)
train_indices = []
test_indices = []
for category in np.unique(y):
    category_indices = np.flatnonzero(y == category)
    random_generator.shuffle(category_indices)
    test_count = max(1, int(round(len(category_indices) * 0.2)))
    test_indices.extend(category_indices[:test_count])
    train_indices.extend(category_indices[test_count:])
random_generator.shuffle(train_indices)
random_generator.shuffle(test_indices)

X_train, X_test = X[train_indices], X[test_indices]
y_train, y_test = y[train_indices], y[test_indices]

# Scale with training statistics only.
feature_means = X_train.mean(axis=0)
feature_scales = X_train.std(axis=0)
feature_scales[feature_scales == 0] = 1.0
X_train = (X_train - feature_means) / feature_scales
X_test = (X_test - feature_means) / feature_scales

print("Feature matrix:", X.shape)
print("Categories:", np.unique(y))
print("Training rows:", len(y_train), "Testing rows:", len(y_test))

Feature matrix: (46344, 300)
Categories: ['hpc' 'iot' 'networks' 'nlp' 'security' 'vision']
Training rows: 37074 Testing rows: 9270


In [18]:
def softmax(scores):
    # Subtract the row maximum to prevent exponential overflow.
    stable_scores = scores - np.max(scores, axis=1, keepdims=True)
    exponentials = np.exp(stable_scores)
    return exponentials / np.sum(exponentials, axis=1, keepdims=True)


def multiclass_cross_entropy(one_hot_labels, probabilities, sample_weights):
    probabilities = np.clip(probabilities, 1e-12, 1.0)
    losses = -np.sum(one_hot_labels * np.log(probabilities), axis=1)
    return np.sum(sample_weights * losses) / np.sum(sample_weights)


def train_multiclass_logistic_regression(
    X, labels, classes, learning_rate=0.1, epochs=1000
):
    sample_count, feature_count = X.shape
    class_count = len(classes)
    weights = np.zeros((feature_count, class_count))
    bias = np.zeros(class_count)
    losses = []

    label_ids = np.searchsorted(classes, labels)
    one_hot_labels = np.zeros((sample_count, class_count))
    one_hot_labels[np.arange(sample_count), label_ids] = 1.0

    # Balance the six categories without using a built-in classifier.
    class_counts = np.bincount(label_ids, minlength=class_count)
    class_weights = sample_count / (class_count * class_counts)
    sample_weights = class_weights[label_ids]

    for epoch in range(epochs):
        logits = np.dot(X, weights) + bias
        probabilities = softmax(logits)

        weighted_error = sample_weights[:, None] * (probabilities - one_hot_labels)
        dw = (1 / sample_count) * np.dot(X.T, weighted_error)
        db = (1 / sample_count) * np.sum(weighted_error, axis=0)

        weights -= learning_rate * dw
        bias -= learning_rate * db
        losses.append(
            multiclass_cross_entropy(one_hot_labels, probabilities, sample_weights)
        )

    return weights, bias, losses

In [19]:
classes = np.unique(y_train)

# Train one multiclass softmax model, not independent sigmoid models.
multiclass_weights, multiclass_bias, losses = train_multiclass_logistic_regression(
    X_train, y_train, classes
)

test_logits = np.dot(X_test, multiclass_weights) + multiclass_bias
test_probabilities = softmax(test_logits)
predicted_categories = classes[np.argmax(test_probabilities, axis=1)]

print("Trained multiclass softmax model")
print("Probability row sums:", test_probabilities[:3].sum(axis=1))
print("First prediction:", y_test[0], "->", predicted_categories[0])

Trained multiclass softmax model
Probability row sums: [1. 1. 1.]
First prediction: vision -> vision


In [20]:
accuracy = np.mean(predicted_categories == y_test)
print(f"Accuracy: {accuracy:.4f}")

for category in classes:
    true_positive = np.sum((y_test == category) & (predicted_categories == category))
    false_positive = np.sum((y_test != category) & (predicted_categories == category))
    false_negative = np.sum((y_test == category) & (predicted_categories != category))

    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    print(f"{category}: precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}")

Accuracy: 0.2589
hpc: precision=0.3164, recall=0.3683, f1=0.3404
iot: precision=0.8131, recall=0.1394, f1=0.2380
networks: precision=0.3201, recall=0.3480, f1=0.3335
nlp: precision=0.4369, recall=0.3903, f1=0.4123
security: precision=0.0181, recall=0.2432, f1=0.0338
vision: precision=0.1221, recall=0.2421, f1=0.1624


## Try your own research-paper text

The next cell converts your title or abstract into the same averaged Word2Vec document vector used by the model. It then applies multiclass softmax, so the displayed scores form one normalized distribution and sum to `1.0`.

A correct probability calculation does not guarantee a correct category for every text. The current Word2Vec features were trained for only one pass, so the evaluation accuracy is the honest measure of model quality.

In [24]:
import re

VOCABULARY_PATH = PROJECT_ROOT / "data" / "processed_data" / "word2vec_vocabulary.csv"
WORD_VECTOR_PATH = PROJECT_ROOT / "data" / "processed_data" / "word_vectors.dat"
VECTOR_SIZE = 300

vocabulary_table = pd.read_csv(VOCABULARY_PATH)
word_to_id = dict(zip(vocabulary_table["word"], vocabulary_table["word_id"]))
word_vectors = np.memmap(
    WORD_VECTOR_PATH,
    mode="r",
    dtype="float32",
    shape=(len(word_to_id), VECTOR_SIZE),
)


def text_to_document_vector(text):
    tokens = re.findall(r"[a-z0-9]+(?:[-'][a-z0-9]+)*", text.lower())
    known_vectors = [
        word_vectors[word_to_id[token]]
        for token in tokens
        if token in word_to_id
    ]
    if not known_vectors:
        raise ValueError("No words from this text were found in the Word2Vec vocabulary")
    return np.mean(known_vectors, axis=0).astype(np.float64)


def predict_category(text, show_probabilities=True):
    document_vector = text_to_document_vector(text)
    scaled_vector = (document_vector - feature_means) / feature_scales
    logits = np.dot(scaled_vector, multiclass_weights) + multiclass_bias
    probabilities = softmax(logits.reshape(1, -1))[0]
    assert np.isclose(probabilities.sum(), 1.0)
    predicted_category = classes[np.argmax(probabilities)]

    print("Text:", text)
    print("Predicted category:", predicted_category)
    if show_probabilities:
        print("Softmax probabilities (sum =", f"{probabilities.sum():.4f}):")
        for category, probability in sorted(
            zip(classes, probabilities), key=lambda item: item[1], reverse=True
        ):
            print(f"  {category}: {probability:.4f}")
    return predicted_category, probabilities

In [52]:
user_text = input("enter user input:")
predict_category(user_text)

Text: Neuromorphic photonic computing with an electro-optic analog memory
Predicted category: security
Softmax probabilities (sum = 1.0000):
  security: 0.9998
  hpc: 0.0002
  networks: 0.0000
  iot: 0.0000
  nlp: 0.0000
  vision: 0.0000


('security',
 array([1.58532754e-04, 7.84016063e-15, 1.57998428e-14, 1.16460086e-17,
        9.99841467e-01, 1.73955605e-20]))